In [39]:
import urllib.request
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt',
    'names.txt'
)

('names.txt', <http.client.HTTPMessage at 0x1c6de6cde00>)

In [40]:
words = open('names.txt', 'r').read().splitlines()

In [41]:
training_size = int(len(words)*0.8)
dev_size = int(len(words)*0.1)
test_size = len(words) - training_size - dev_size

In [42]:
import torch
from torch.utils.data import random_split

In [43]:
g = torch.Generator().manual_seed(2147483647)
train_set, dev_set, test_set = random_split(words, [training_size, dev_size, test_size], generator = g)

In [44]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

In [45]:
xtrain = []
ytrain = []
for w in train_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xtrain.append(ix1)
        ytrain.append(ix2)

xtrain = torch.tensor(xtrain)
ytrain = torch.tensor(ytrain)

In [46]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), dtype = torch.float32, generator = g, requires_grad = True)

In [47]:
import torch.nn.functional as F
xenc_train = F.one_hot(xtrain, num_classes = 27).float()

In [48]:
for i in range(100):
    logits = xenc_train @ W
    counts = logits.exp()
    probs = counts/counts.sum(1, keepdims = True)
    loss = (-probs[torch.arange(len(xtrain)), ytrain].log()).mean() + 0.01*(W**2).mean()

    W.grad = None
    loss.backward()

    W.data += -50*W.grad

print(loss)

tensor(2.4904, grad_fn=<AddBackward0>)


In [49]:
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes = 27).float()
        logits = xenc @ W
        counts = logits.exp()
        probs = counts/counts.sum(1, keepdims = True)
        ix = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    word = ''.join(out)
    print(word)

cexze.
momakurailezityha.
konimittain.
llayn.
ka.


In [50]:
x_dev = []
y_dev = []
for w in dev_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        x_dev.append(ix1)
        y_dev.append(ix2)

x_dev = torch.tensor(x_dev)
y_dev = torch.tensor(y_dev)

In [51]:
xenc_dev = F.one_hot(x_dev, num_classes = 27).float()

In [52]:
logits = xenc_dev @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)
loss = (-probs[torch.arange(len(x_dev)),y_dev].log()).mean()
loss

tensor(2.4747, grad_fn=<MeanBackward0>)

In [53]:
x_test = []
y_test = []
for w in test_set:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        x_test.append(ix1)
        y_test.append(ix2)

x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test)

In [54]:
xenc_test = F.one_hot(x_test, num_classes = 27).float()

In [55]:
logits = xenc_test @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims = True)
loss = (-probs[torch.arange(len(x_test)), y_test].log()).mean()
loss

tensor(2.4744, grad_fn=<MeanBackward0>)